# Receptor Morse por video
Tarea 1, Sistemas de Comunicaciones 2026 · Julián Alfaro y Matías Vargas

Ejecuta las celdas en orden. La celda 2 pide el video de **PARIS PARIS a 0,5 m**; la celda 3 pide el video a decodificar y se puede repetir con cada video.

In [ ]:
# CELDA 1: programa del receptor (mismo método del informe)
import cv2, numpy as np, matplotlib.pyplot as plt
from google.colab import files
MORSE = dict(zip(".- -... -.-. -.. . ..-. --. .... .. .--- -.- .-.. -- -. --- .--. --.- .-. ... - ..- ...- .-- -..- -.-- --..".split(), "ABCDEFGHIJKLMNOPQRSTUVWXYZ"))

def senal(ruta, roi=(24, 12), escala=8, centro=None):
    """Video -> I[n] (promedio de la ROI) y t[n] (hora de cada cuadro). centro=(x, y) elige la ROI a mano."""
    cap, F, t = cv2.VideoCapture(ruta), [], []
    while (ok := cap.read())[0]:
        g = cv2.cvtColor(ok[1], cv2.COLOR_BGR2GRAY)
        F.append(cv2.resize(g, None, fx=1/escala, fy=1/escala, interpolation=cv2.INTER_AREA))
        t.append(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000)
    F = np.array(F, float)
    x, y = centro or np.unravel_index(cv2.GaussianBlur(F.std(0), (5, 5), 0).argmax(), F.shape[1:])[::-1]  # ROI donde más varía el brillo
    y0, x0 = max(0, y - roi[1]//2), max(0, x - roi[0]//2)
    return F[:, y0:y0+roi[1], x0:x0+roi[0]].mean((1, 2)), np.array(t)

def otsu(I):
    """Umbral que más separa los dos grupos (encendido y apagado)."""
    p = np.histogram(I, 256, (0, 256))[0] / len(I); w = p.cumsum(); m = (p * np.arange(256)).cumsum()
    s = (m[-1]*w - m)**2 / (w*(1 - w) + 1e-12); k = np.flatnonzero(s >= s.max()*(1 - 1e-6))
    return (k.min() + k.max())/2 + 0.5

def rachas(I, t, Iu, margen=2.5):
    """Tramos con el mismo estado y su duración medida con la hora de los cuadros.
    El mensaje es el grupo de pulsos entre los márgenes apagados (pausas de más de 'margen' s)."""
    b = (I > Iu).astype(int); c = np.r_[0, np.flatnonzero(np.diff(b)) + 1, len(b) - 1]
    G = [[]]
    for i, j in zip(c[:-1], c[1:]):
        if b[i] == 0 and t[j] - t[i] >= margen: G.append([])
        else: G[-1].append((b[i], t[j] - t[i]))
    G = [g[next(k for k, r in enumerate(g) if r[0]):len(g) - next(k for k, r in enumerate(g[::-1]) if r[0])] for g in G if any(r[0] for r in g)]
    return max(G, key=lambda g: sum(r[0] for r in g), default=[])

def texto(R, T):
    """ON: <2T punto, 2T a 6T raya. OFF: desde 2T fin de letra, desde 5T fin de palabra."""
    s, l = "", ""
    for nivel, d in R:
        if nivel: l += "." if d < 2*T else "-" if d < 6*T else "#"
        elif d >= 2*T: s += MORSE.get(l, "?") + (" " if d >= 5*T else ""); l = ""
    return s + MORSE.get(l, "?")

def decisiones(R, T, n=12):
    """Muestra un tramo de rachas con su duración en T y la decisión tomada."""
    for k, (nivel, d) in enumerate(R[:n], 1):
        x = d / T; dec = ("punto" if x < 2 else "raya" if x < 6 else "inválido") if nivel else ("entre símbolos" if x < 2 else "entre letras" if x < 5 else "entre palabras")
        print(f"{k:3d}  {'ON ' if nivel else 'OFF'}  {d:.3f} s  {x:5.2f} T  {dec}")

def cer(ref, rec):
    """Distancia de Levenshtein (Wagner-Fischer) dividida por el largo de la frase."""
    d = list(range(len(rec) + 1))
    for i, a in enumerate(ref, 1):
        p, d[0] = d[0], i
        for j, b in enumerate(rec, 1): p, d[j] = d[j], min(d[j] + 1, d[j-1] + 1, p + (a != b))
    return d[-1], d[-1] / len(ref)

def graficar(t, I, Iu, s):
    plt.figure(figsize=(11, 3)); plt.plot(t - t[0], I, ".-", ms=3); plt.axhline(Iu, color="orange", ls="--")
    plt.title(f"Texto reconstruido: {s}"); plt.xlabel("Tiempo [s]"); plt.ylabel("I[n]"); plt.grid(alpha=.3); plt.show()
print("Programa cargado.")


In [ ]:
# CELDA 2: calibración con PARIS PARIS a 0,5 m (una sola vez)
ROI, ESCALA = (3, 3), 4          # flash: (3, 3), 4   |   pantalla: (24, 12), 8
CENTRO = None                    # o (x, y) para elegir la ROI a mano
I, t = senal(list(files.upload())[0], ROI, ESCALA, CENTRO)
Iu = otsu(I); R = rachas(I, t, Iu); T = sum(d for _, d in R) / 93      # PARIS PARIS dura 93 T
print(f"fs = {(len(t)-1)/(t[-1]-t[0]):.2f} Hz   Umbral = {Iu:.1f}   T = {T*1000:.1f} ms   texto: {texto(R, T)}"); graficar(t, I, Iu, texto(R, T))


In [ ]:
# CELDA 3: decodificar un video (usa el umbral y T de la calibración)
FRASE = "SIS COM"                # solo para calcular la CER
CENTRO = None                    # o (x, y) para elegir la ROI a mano
I, t = senal(list(files.upload())[0], ROI, ESCALA, CENTRO)
R = rachas(I, t, Iu); s = texto(R, T); e, c = cer(FRASE, s)
print(f"fs = {(len(t)-1)/(t[-1]-t[0]):.2f} Hz   duración del mensaje = {sum(d for _, d in R):.2f} s")
print(f"Texto recibido: '{s}'   S+D+I = {e}   CER = {c:.4f}   exacta: {s == FRASE}"); graficar(t, I, Iu, s); decisiones(R, T)


In [ ]:
# CELDA 4: espectro, PSD (periodograma) y B95 del último video
fs = (len(t) - 1) / (t[-1] - t[0]); x = I - I.mean(); X = np.fft.rfft(x); f = np.fft.rfftfreq(len(x), 1/fs)
P = 2*abs(X)**2 / (fs*len(x)); P[0] = 0; A = np.r_[0, np.cumsum((P[1:] + P[:-1])/2 * np.diff(f))]
B95 = f[np.searchsorted(A, 0.95*A[-1])]; pico = f[f > 1][abs(X)[f > 1].argmax()]
fig, ax = plt.subplots(1, 2, figsize=(12, 3)); ax[0].plot(f, 2*abs(X)/len(x)); ax[0].set_title(f"Espectro (pico {pico:.2f} Hz)")
ax[1].plot(f, P); ax[1].axvline(B95, color="orange", ls="--"); ax[1].set_title(f"PSD, B95 = {B95:.2f} Hz"); plt.show()
